# CNN Hands-On Lab: 이미지 분류, 전이학습, 객체 탐지

이 노트북은 합성곱 신경망(CNN)이 실제 컴퓨터 비전 문제에서 어떻게 쓰이는지,
세 가지 실습을 통해 점점 더 어려운 질문에 답해가며 배웁니다.

| Part | 질문 | 핵심 아이디어 |
|---|---|---|
| **Part 1. 이미지 분류** | "이 이미지에 뭐가 있지?" | 사전학습된 CNN이 feature를 뽑아 카테고리를 예측 |
| **Part 2. 전이학습** | "기존 CNN을 새로운 문제에 어떻게 맞출까?" | 적은 데이터로도 사전학습된 지식을 재활용 |
| **Part 3. 객체 탐지** | "어떤 물체가, 어디에 있지?" | 한 번의 추론으로 여러 물체의 종류와 위치를 동시에 예측 |

**학습 목표**
- 사전학습된(pretrained) 모델과 ImageNet의 의미를 이해한다.
- Feature Extraction과 Fine-tuning의 차이를 직접 실험으로 비교한다.
- Image Classification과 Object Detection의 차이, 그리고 confidence/NMS 같은
  객체 탐지 고유의 개념을 이해한다.

**실행 환경:** Google Colab (`런타임 > 런타임 유형 변경`에서 GPU 선택을 권장합니다 —
GPU가 없어도 동작하지만 Part 2 학습이 더 오래 걸립니다). 전체 노트북을 처음부터
끝까지 실행하는 데 GPU 기준 약 10~15분 정도 걸립니다.

## Part 0. 환경 설정

아래 셀들은 모든 Part에서 공통으로 쓰는 라이브러리 설치/임포트, device 설정,
random seed 고정입니다. **반드시 가장 먼저 한 번 실행**하세요 — 이후 Part 1, 2, 3은
이 셀들만 실행되어 있으면 서로 독립적으로 실행할 수 있습니다.

In [ ]:
# Ultralytics(YOLO)는 Colab에 기본 설치되어 있지 않으므로 설치합니다.
# torch/torchvision은 Colab에 이미 설치되어 있어 별도 설치가 필요 없습니다.
!pip install -q ultralytics

In [ ]:
import os
import time
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from PIL import Image

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

### 공통 유틸리티 함수

이미지를 내려받는 함수와, 분류 결과를 시각화하는 함수를 미리 정의해둡니다.
Part 1에서 바로 쓰고, Part 2에서도 재사용합니다.

In [ ]:
def download_image(url, filename):
    """주어진 URL의 이미지를 현재 디렉터리에 내려받고 PIL Image로 반환합니다."""
    urllib.request.urlretrieve(url, filename)
    return Image.open(filename).convert("RGB")


def show_topk_predictions(image, labels, probs, title=""):
    """이미지와 Top-k 예측 결과(막대그래프)를 나란히 보여줍니다."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(image)
    axes[0].axis("off")
    axes[0].set_title(title if title else "Input Image")

    y_pos = np.arange(len(labels))
    axes[1].barh(y_pos, probs, color="steelblue")
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(labels)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("Probability")
    axes[1].set_xlim(0, 1)
    axes[1].set_title("Top-{} Predictions".format(len(labels)))

    plt.tight_layout()
    plt.show()

## Part 1. 사전학습된 CNN으로 이미지 분류하기

### 목표

추가 학습 없이, **이미 ImageNet으로 학습된 CNN(ResNet18)**이 새로운 이미지를 보고
어떻게 카테고리를 예측하는지 이해합니다.

**진행 순서:** 사전학습된 모델을 불러온다 → 이미지를 모델이 원하는 형태로
전처리한다 → 추론한다 → 점수(logit)를 확률로 바꿔 Top-5 예측을 확인한다.

### 개념 설명

- **Pretrained weights(사전학습 가중치)**: 이 모델은 ImageNet이라는 대규모 데이터셋
  (약 128만 장, 1,000개 클래스)으로 이미 학습이 끝난 가중치를 가지고 있습니다. 우리는
  이 가중치를 그대로 가져다 쓸 뿐, 추가로 학습시키지 않습니다.
- **ImageNet**: 동물, 사물, 음식 등 1,000개 카테고리로 이루어진 이미지 분류의
  표준 벤치마크 데이터셋입니다. 모델이 "아는" 카테고리는 이 1,000개뿐입니다 —
  학습 때 본 적 없는 범주(예: 특정 브랜드 로고)는 비슷한 1,000개 클래스 중 하나로
  **억지로** 분류됩니다.
- **Logit vs. Probability**: 모델의 마지막 층은 클래스마다 하나씩, 총 1,000개의
  실수값(logit)을 출력합니다. 이 값 자체는 "확률"이 아니라서 음수일 수도, 합이
  1이 아닐 수도 있습니다. **Softmax** 함수를 적용해야 모든 값이 0~1 사이이고
  합이 1이 되는 진짜 확률로 바뀝니다.
- **`model.eval()`**: 학습(train) 모드와 평가(eval) 모드에서 다르게 동작하는 층들
  (Dropout, BatchNorm 등)이 있습니다. 추론만 할 때는 반드시 `eval()`을 호출해서
  이런 층들이 "학습 때처럼 무작위로 끄고 켜기"를 하지 않고 일관되게 동작하도록
  해야 합니다.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

# IMAGENET1K_V1: torchvision 공식 API로 ImageNet 사전학습 가중치를 받아옵니다.
weights = ResNet18_Weights.IMAGENET1K_V1
clf_model = resnet18(weights=weights).to(device)
clf_model.eval()  # 추론 전용 모드

imagenet_categories = weights.meta["categories"]
print("ImageNet 클래스 개수:", len(imagenet_categories))
print("예시:", imagenet_categories[:5])

### 전처리(Preprocessing)

CNN은 학습할 때 본 것과 **똑같은 방식**으로 전처리된 입력을 기대합니다. 크기를
224×224로 맞추고, 픽셀값을 ImageNet 학습 때 사용한 평균/표준편차로 정규화해야
합니다. `weights.transforms()`를 쓰면 이 전처리를 모델 학습 때와 정확히
동일하게 재현해줍니다 — 직접 숫자를 외울 필요가 없습니다.

In [ ]:
preprocess = weights.transforms()
print(preprocess)

### 샘플 이미지 불러오기

공개된 예제 이미지 두 장을 내려받습니다 (버스/사람들이 있는 거리 사진,
두 사람이 있는 사진). 아래에서 직접 이미지를 업로드해서 테스트할 수도 있습니다.

In [ ]:
ASSET_BASE_URL = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets"

sample_images = {
    "bus.jpg": download_image(f"{ASSET_BASE_URL}/bus.jpg", "bus.jpg"),
    "zidane.jpg": download_image(f"{ASSET_BASE_URL}/zidane.jpg", "zidane.jpg"),
}

fig, axes = plt.subplots(1, len(sample_images), figsize=(10, 5))
for ax, (name, img) in zip(axes, sample_images.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
# (선택) 직접 이미지를 업로드해서 테스트해보고 싶다면 이 셀을 실행하세요.
# Google Colab에서만 동작합니다 -- 로컬 환경이나 Colab이 아닌 곳에서는 건너뜁니다.
try:
    from google.colab import files

    uploaded = files.upload()
    for filename in uploaded:
        sample_images[filename] = Image.open(filename).convert("RGB")
    print("업로드된 이미지:", list(uploaded.keys()))
except ImportError:
    print("Google Colab 환경이 아니므로 업로드를 건너뜁니다. 기본 샘플 이미지로 계속합니다.")

### 추론 및 Top-5 예측

전처리 → 모델 추론 → Softmax → Top-5 추출까지 한 번에 수행하는 함수를 만들고,
샘플 이미지 각각에 대해 실행해봅니다.

In [ ]:
def classify_image(image, top_k=5):
    x = preprocess(image).unsqueeze(0).to(device)  # (1, 3, 224, 224)
    with torch.no_grad():
        logits = clf_model(x)                       # (1, 1000) -- 아직 확률이 아님
        probs = torch.softmax(logits, dim=1)[0]      # Softmax로 진짜 확률로 변환
    top_probs, top_idx = torch.topk(probs, top_k)
    labels = [imagenet_categories[i] for i in top_idx.cpu()]
    return labels, top_probs.cpu().numpy()


for name, img in sample_images.items():
    labels, probs = classify_image(img, top_k=5)
    print(f"\n[{name}] Top-5 예측:")
    for label, p in zip(labels, probs):
        print(f"  {label:25s} {p:.2%}")
    show_topk_predictions(img, labels, probs, title=name)

**참고할 점:** 이 모델은 ImageNet의 **1,000개 클래스 중 하나**만 답할 수 있습니다.
예를 들어 "스쿨버스"는 ImageNet 클래스에 있지만 "우리 학교 통학버스"처럼 더
구체적인 범주는 없습니다 — 학습 때 본 적 없는 카테고리는 절대 정답으로 나올 수
없고, 1,000개 중 가장 비슷해 보이는 클래스로만 분류됩니다. Part 2에서는 바로 이
한계를 **전이학습**으로 어떻게 극복하는지 다룹니다.

In [ ]:
# 다음 Part에서 새 모델 인스턴스를 쓸 것이므로, 여기서 쓴 모델을 메모리에서 정리합니다.
del clf_model
if device.type == "cuda":
    torch.cuda.empty_cache()

## Part 2. 전이학습(Transfer Learning): Feature Extraction vs. Fine-tuning

### 목표

Part 1의 모델은 ImageNet 1,000개 클래스만 압니다. **고양이/개 37개 품종**처럼 더
구체적인 새 문제에 맞추려면 어떻게 해야 할까요? 이번 Part에서는 처음부터 다시
학습시키는 대신, ImageNet 사전학습 가중치를 "재활용"하는 두 가지 방법 —
**Feature Extraction**과 **Fine-tuning** — 을 직접 구현하고 성능을 비교합니다.

**데이터셋:** [Oxford-IIIT Pet](https://www.robots.ox.ac.uk/~vgg/data/pets/) —
고양이/개 37개 품종, 약 7,400장의 실제 사진. `torchvision.datasets.OxfordIIITPet`으로
자동으로 내려받습니다.

### 개념 설명

- **왜 사전학습 모델을 재사용할 수 있을까?** CNN의 앞쪽 층들은 선, 질감, 색 같은
  **일반적인 시각 특징**을 뽑아내도록 학습됩니다. 이런 특징은 "고양이냐 개냐"뿐
  아니라 거의 모든 이미지 문제에 공통으로 유용합니다. 뒤쪽 층(특히 마지막
  FC layer)만 "ImageNet 1,000개 클래스 중 하나를 고르는" 역할을 하므로, 이 부분만
  새 문제(37개 품종)에 맞게 바꿔주면 됩니다.
- **마지막 FC layer를 교체하는 이유**: 원래 ResNet18의 마지막 층은 1,000개
  출력을 냅니다. 우리는 37개 품종을 구분해야 하므로, 이 층을 출력 37개짜리
  새 `nn.Linear` 층으로 바꿔야 합니다 (이 새 층은 무작위로 초기화되어 학습이
  필요합니다).
- **Feature Extraction**: 새로 추가한 FC layer **만** 학습시키고, 나머지(backbone)는
  전부 얼립니다(freeze) — `requires_grad = False`로 설정해서 역전파 때 이 파라미터들의
  gradient를 계산도, 업데이트도 하지 않습니다. 이미 잘 학습된 특징 추출기를 그대로
  믿고 쓰는 방식입니다.
- **Fine-tuning**: FC layer뿐 아니라 뒤쪽 일부 층(`layer4`)까지 **함께** 학습시킵니다.
  이미 좋은 특징을 가지고 있는 가중치를 완전히 망가뜨리지 않도록, 새로 추가한
  FC layer보다 **더 작은 learning rate**로 조심스럽게 업데이트합니다.

### 데이터 준비

실행 시간을 짧게 유지하기 위해 전체 데이터셋 중 일부(`SUBSET_SIZE`)만 사용하고,
epoch 수도 적게(`NUM_EPOCHS`) 설정합니다. 학습용 이미지에는 **data augmentation**
(뒤집기 등)을 적용해 과적합을 줄이고, 검증용 이미지는 항상 같은 방식(결정적
전처리)으로만 변환합니다 — 검증 성능이 "운 좋은 변형" 때문에 흔들리지 않도록요.

In [ ]:
# 실행 시간 조절용 하이퍼파라미터 -- 필요하면 값을 바꿔서 실행 시간/성능을 조절할 수 있습니다.
SUBSET_SIZE = 800     # 전체(~3,680장) 중 이번 실습에 쓸 이미지 수
TRAIN_RATIO = 0.8      # 그중 학습:검증 비율
BATCH_SIZE = 32
NUM_EPOCHS = 4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

from torchvision import transforms as T

train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [ ]:
from torch.utils.data import Subset, DataLoader

# trainval split을 내려받고(download=True는 이미 받아져 있으면 다시 받지 않습니다),
# train/val 각각 다른 transform을 적용한 두 인스턴스를 만든 뒤 같은 인덱스로 나눕니다.
train_dataset_full = torchvision.datasets.OxfordIIITPet(
    root="./data", split="trainval", transform=train_transform, download=True
)
val_dataset_full = torchvision.datasets.OxfordIIITPet(
    root="./data", split="trainval", transform=val_transform, download=True
)

pet_classes = train_dataset_full.classes
print("품종(클래스) 개수:", len(pet_classes))
print("예시:", pet_classes[:5])

rng = np.random.default_rng(SEED)
n_available = len(train_dataset_full)
n_use = min(SUBSET_SIZE, n_available)
indices = rng.permutation(n_available)[:n_use]
n_train = int(n_use * TRAIN_RATIO)
train_indices, val_indices = indices[:n_train], indices[n_train:]

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)
print(f"학습 이미지: {len(train_dataset)}장, 검증 이미지: {len(val_dataset)}장")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# 학습 데이터 샘플 몇 장을 확인해봅니다 (정규화를 되돌려서 원래 색으로 보여줍니다).
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, idx in zip(axes, range(5)):
    img, label = train_dataset[idx]
    ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
    ax.set_title(pet_classes[label], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

### 학습/평가 공통 함수

Feature Extraction과 Fine-tuning 모두 **똑같은 학습 루프**를 쓰므로, 한 번만
정의해서 재사용합니다.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


def run_experiment(name, model, optimizer, num_epochs=NUM_EPOCHS):
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start_time = time.time()
    for epoch in range(num_epochs):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = evaluate(model, val_loader, criterion)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        print(f"[{name}] epoch {epoch + 1}/{num_epochs}  "
              f"train_loss={tr_loss:.4f} train_acc={tr_acc:.2%}  "
              f"val_loss={va_loss:.4f} val_acc={va_acc:.2%}")
    elapsed = time.time() - start_time
    return history, elapsed

### 실험 A: Feature Extraction

새 ResNet18 인스턴스를 ImageNet 사전학습 가중치로 만든 뒤, **모든 파라미터를
얼리고** 마지막 FC layer만 37개 클래스용으로 교체해서 그 층만 학습시킵니다.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights  # Part 1에서도 쓴 것과 동일한 모델/가중치

weights = ResNet18_Weights.IMAGENET1K_V1

model_fe = resnet18(weights=weights).to(device)

for param in model_fe.parameters():       # 1. 모든 파라미터를 얼립니다
    param.requires_grad = False

model_fe.fc = nn.Linear(model_fe.fc.in_features, len(pet_classes)).to(device)  # 2. FC layer 교체 (새 층은 기본적으로 학습 가능)

trainable_params_fe = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
total_params_fe = sum(p.numel() for p in model_fe.parameters())
print(f"학습 가능한 파라미터: {trainable_params_fe:,} / 전체 {total_params_fe:,} "
      f"({trainable_params_fe / total_params_fe:.2%})")

optimizer_fe = torch.optim.Adam(model_fe.fc.parameters(), lr=1e-3)  # 3. FC layer만 옵티마이저에 전달

history_fe, time_fe = run_experiment("Feature Extraction", model_fe, optimizer_fe)

### 실험 B: Fine-tuning

**새 ResNet18 인스턴스**를 다시 만듭니다 (실험 A에서 이미 학습된 모델을 재사용하지
않습니다). FC layer를 교체하는 것까지는 같지만, 이번에는 `layer4`(마지막 conv
블록)도 함께 학습시킵니다 — 단, 사전학습된 가중치가 갑자기 망가지지 않도록 FC
layer보다 10배 작은 learning rate를 적용합니다.

In [ ]:
model_ft = resnet18(weights=weights).to(device)  # 사전학습 가중치로 "새" 인스턴스

for param in model_ft.parameters():
    param.requires_grad = False  # 일단 전부 얼린 뒤

model_ft.fc = nn.Linear(model_ft.fc.in_features, len(pet_classes)).to(device)

for param in model_ft.layer4.parameters():  # layer4만 다시 학습 가능하게 풀어줍니다
    param.requires_grad = True
for param in model_ft.fc.parameters():
    param.requires_grad = True

trainable_params_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
total_params_ft = sum(p.numel() for p in model_ft.parameters())
print(f"학습 가능한 파라미터: {trainable_params_ft:,} / 전체 {total_params_ft:,} "
      f"({trainable_params_ft / total_params_ft:.2%})")

# 서로 다른 learning rate를 적용하기 위해 optimizer의 parameter group을 나눕니다.
optimizer_ft = torch.optim.Adam([
    {"params": model_ft.layer4.parameters(), "lr": 1e-4},  # 사전학습된 층: 조심스럽게
    {"params": model_ft.fc.parameters(), "lr": 1e-3},       # 새로 추가한 층: 과감하게
])

history_ft, time_ft = run_experiment("Fine-tuning", model_ft, optimizer_ft)

### 결과 시각화 및 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, history in [("Feature Extraction", history_fe), ("Fine-tuning", history_ft)]:
    axes[0].plot(history["train_loss"], "--", label=f"{name} (train)")
    axes[0].plot(history["val_loss"], "-", label=f"{name} (val)")
    axes[1].plot(history["train_acc"], "--", label=f"{name} (train)")
    axes[1].plot(history["val_acc"], "-", label=f"{name} (val)")

axes[0].set_title("Loss over Epochs")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend(fontsize=8)

axes[1].set_title("Accuracy over Epochs")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": ["Feature Extraction", "Fine-tuning"],
    "Trainable Parameters": [trainable_params_fe, trainable_params_ft],
    "Training Time (s)": [round(time_fe, 1), round(time_ft, 1)],
    "Validation Accuracy": [f"{history_fe['val_acc'][-1]:.2%}", f"{history_ft['val_acc'][-1]:.2%}"],
})
comparison

### 예측 예시 확인

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, num_images=5, title=""):
    model.eval()
    fig, axes = plt.subplots(1, num_images, figsize=(13, 3))
    indices = np.random.default_rng(SEED).choice(len(dataset), num_images, replace=False)
    for ax, idx in zip(axes, indices):
        img, label = dataset[idx]
        pred = model(img.unsqueeze(0).to(device)).argmax(1).item()
        ax.imshow(denormalize(img).permute(1, 2, 0).numpy())
        color = "green" if pred == label else "red"
        ax.set_title(f"true: {pet_classes[label]}\npred: {pet_classes[pred]}", fontsize=8, color=color)
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_predictions(model_fe, val_dataset, title="Feature Extraction")
show_predictions(model_ft, val_dataset, title="Fine-tuning")

In [ ]:
# Part 3으로 넘어가기 전에 메모리를 정리합니다.
del model_fe, model_ft
if device.type == "cuda":
    torch.cuda.empty_cache()

## Part 3. 객체 탐지(Object Detection): YOLO

### 목표

Part 1, 2는 모두 **"이 이미지 전체가 무엇인가?"** 를 답하는 **분류(classification)**
문제였습니다. 하지만 실제로는 "이미지 안에 어떤 물체들이 있고, 그것들이
**어디에** 있는지"도 알아야 하는 경우가 많습니다 (자율주행, CCTV, 재고 관리 등).
이것이 **객체 탐지(object detection)** 입니다. 이번 Part에서는 사전학습된
**YOLO(You Only Look Once)** 모델로 이미지(그리고 마지막에는 여러분의 웹캠)에서
객체를 탐지하고, 분류와는 다른 고유 개념들(confidence, NMS)을 살펴봅니다.

### 개념 설명

- **분류 vs 탐지**: 분류는 이미지 하나당 레이블 하나(또는 Top-K 확률)를
  출력합니다. 탐지는 이미지 하나에서 **여러 개의 bounding box**를 출력하는데,
  각 box마다 (1) 위치(좌표), (2) 클래스, (3) **confidence(그 box에 물체가 있다는
  확신도)** 세 가지를 함께 예측합니다.
- **Confidence threshold**: 모델은 실제로는 수천 개의 후보 box를 평가하지만,
  confidence가 낮은 대부분은 버립니다. threshold를 낮추면 더 많은(그러나 틀릴
  가능성도 높은) box가 남고, 높이면 확실한 것만 남습니다 — **재현율(recall)과
  정밀도(precision) 사이의 트레이드오프**입니다.
- **NMS (Non-Maximum Suppression)**: 같은 물체 주변에 겹치는 box가 여러 개
  나올 수 있습니다 (같은 개를 약간 다른 위치/크기로 여러 번 탐지). NMS는 겹치는
  정도(**IoU**, Intersection over Union)가 특정 threshold 이상인 box들 중
  confidence가 가장 높은 하나만 남기고 나머지를 제거합니다.

### 모델 준비

`ultralytics` 라이브러리는 Part 0에서 이미 설치했습니다. `YOLO("yolo11n.pt")`를
처음 호출하면 사전학습된 가중치를 자동으로 내려받습니다 ("n"은 nano — 가장
작고 빠른 버전).

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolo11n.pt")
print("클래스 개수:", len(yolo_model.names))
print("예시 클래스:", list(yolo_model.names.values())[:10])

### 이미지에서 객체 탐지

Part 1에서 썼던 것과 같은 샘플 이미지(bus.jpg, zidane.jpg)로 탐지를 실행합니다.
`model.predict(...)`를 호출하면 이미지마다 하나씩 `Results` 객체가 담긴 리스트가
돌아옵니다. `.plot()`은 bounding box가 그려진 이미지를 바로 만들어줍니다.

In [ ]:
ASSET_BASE_URL = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets"

detection_images = {
    "bus.jpg": download_image(f"{ASSET_BASE_URL}/bus.jpg", "bus.jpg"),
    "zidane.jpg": download_image(f"{ASSET_BASE_URL}/zidane.jpg", "zidane.jpg"),
}

# Colab에서는 직접 업로드한 이미지로 바꿔서 실행해볼 수 있습니다.
try:
    from google.colab import files
    print("이미지를 업로드하면 아래 탐지에 사용됩니다 (취소하려면 Cancel upload를 누르세요).")
    uploaded = files.upload()
    for filename in uploaded:
        detection_images[filename] = filename
except ImportError:
    print("Google Colab 환경이 아니므로 업로드를 건너뜁니다. 기본 샘플 이미지로 계속합니다.")

detection_results = yolo_model.predict(source=list(detection_images.values()), conf=0.25, verbose=False)

fig, axes = plt.subplots(1, len(detection_results), figsize=(13, 6))
if len(detection_results) == 1:
    axes = [axes]
for ax, name, result in zip(axes, detection_images.keys(), detection_results):
    ax.imshow(result.plot()[:, :, ::-1])  # BGR -> RGB
    ax.set_title(f"{name}  ({len(result.boxes)} detections)")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 원본 출력 자세히 보기

`.plot()`은 보기 편하게 그려줄 뿐, 실제로 모델이 반환하는 값은 `result.boxes`에
들어 있습니다. 각 box의 좌표(`xyxy`: 좌상단 x,y와 우하단 x,y), confidence, 클래스
번호를 직접 확인해봅니다.

In [ ]:
first_result = detection_results[0]
print(f"'{list(detection_images.keys())[0]}'에서 탐지된 객체 {len(first_result.boxes)}개:\n")
for box in first_result.boxes:
    xyxy = box.xyxy[0].tolist()
    conf = box.conf.item()
    cls_id = int(box.cls.item())
    class_name = first_result.names[cls_id]
    print(f"  {class_name:12s}  confidence={conf:.2%}  box=({xyxy[0]:.0f}, {xyxy[1]:.0f}, {xyxy[2]:.0f}, {xyxy[3]:.0f})")

### Confidence threshold 비교

같은 이미지에 서로 다른 confidence threshold를 적용해 탐지 결과가 어떻게
달라지는지 비교합니다. threshold가 낮을수록 더 많은(약한 확신의) box가
살아남습니다.

In [ ]:
thresholds = [0.1, 0.25, 0.5, 0.8]
sample_image_path = list(detection_images.values())[0]

fig, axes = plt.subplots(1, len(thresholds), figsize=(16, 5))
for ax, conf_thresh in zip(axes, thresholds):
    result = yolo_model.predict(source=sample_image_path, conf=conf_thresh, verbose=False)[0]
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"conf >= {conf_thresh}\n({len(result.boxes)} detections)")
    ax.axis("off")
plt.tight_layout()
plt.show()

### NMS(IoU threshold)의 효과

`iou` 파라미터는 NMS가 "같은 물체"로 판단해 하나로 합칠 겹침 정도(threshold)를
결정합니다. **낮추면** 조금만 겹쳐도 중복으로 간주해 적극적으로 box를 제거하고,
**높이면** 많이 겹쳐야만 제거하므로 (극단적으로 1.0에 가까우면 거의 제거하지
않아) 같은 물체 주변에 중복된 box가 여러 개 남을 수 있습니다.

In [ ]:
iou_thresholds = [0.3, 0.7, 0.99]

fig, axes = plt.subplots(1, len(iou_thresholds), figsize=(14, 5))
for ax, iou_thresh in zip(axes, iou_thresholds):
    result = yolo_model.predict(source=sample_image_path, conf=0.1, iou=iou_thresh, verbose=False)[0]
    ax.imshow(result.plot()[:, :, ::-1])
    ax.set_title(f"iou threshold = {iou_thresh}\n({len(result.boxes)} detections)")
    ax.axis("off")
plt.tight_layout()
plt.show()

### 웹캠으로 직접 탐지해보기

지금까지는 미리 준비된 이미지로 탐지를 실행했습니다. 이번에는 **여러분의
웹캠**으로 찍은 사진에 똑같은 모델을 돌려봅니다. Colab은 브라우저를 통해
웹캠에 접근하므로, 아래 셀을 실행하면 브라우저가 웹캠 사용 권한을 요청하고,
화면에 "Capture" 버튼이 나타납니다 — 버튼을 누르면 그 순간의 사진이
찍히고, 곧바로 YOLO가 탐지한 결과가 표시됩니다. 매번 "한 번 더
촬영하시겠습니까?"에 `y`를 입력하면 다시 촬영할 수 있어서, 사진을 여러 장
찍어보며 마치 실시간으로 탐지하는 듯한 느낌을 경험할 수 있습니다.

**이 셀은 Google Colab에서 실제 웹캠이 있는 브라우저로 실행해야 동작합니다**
(로컬 Jupyter나 웹캠이 없는 환경에서는 안내 메시지만 출력되고 건너뜁니다).

In [ ]:
try:
    from google.colab.output import eval_js
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("이 데모는 Google Colab에서 브라우저 웹캠에 접근해야 동작합니다. Colab에서 실행해주세요.")
else:
    from IPython.display import display, Javascript
    from base64 import b64decode

    def take_photo(filename="webcam_photo.jpg", quality=0.8):
        """브라우저에서 웹캠 권한을 요청하고, 'Capture' 버튼을 누르면 사진 한 장을 찍어 저장합니다."""
        js = Javascript('''
            async function takePhoto(quality) {
                const div = document.createElement(\'div\');
                const capture = document.createElement(\'button\');
                capture.textContent = \'Capture\';
                div.appendChild(capture);

                const video = document.createElement(\'video\');
                video.style.display = \'block\';
                const stream = await navigator.mediaDevices.getUserMedia({video: true});

                document.body.appendChild(div);
                div.appendChild(video);
                video.srcObject = stream;
                await video.play();

                google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

                await new Promise((resolve) => capture.onclick = resolve);

                const canvas = document.createElement(\'canvas\');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext(\'2d\').drawImage(video, 0, 0);
                stream.getVideoTracks()[0].stop();
                div.remove();
                return canvas.toDataURL(\'image/jpeg\', quality);
            }
        ''')
        display(js)
        data = eval_js(f"takePhoto({quality})")
        binary = b64decode(data.split(",")[1])
        with open(filename, "wb") as f:
            f.write(binary)
        return filename

    while True:
        photo_path = take_photo()
        webcam_result = yolo_model.predict(source=photo_path, conf=0.25, verbose=False)[0]

        plt.figure(figsize=(7, 7))
        plt.imshow(webcam_result.plot()[:, :, ::-1])
        plt.title(f"{len(webcam_result.boxes)} detections")
        plt.axis("off")
        plt.show()

        again = input("한 번 더 촬영하시겠습니까? (y/n): ").strip().lower()
        if again != "y":
            break

### (심화) 실시간 스트리밍 버전

위의 "사진 한 장씩 캡처" 방식은 안정적이지만, 매번 캡처 버튼을 눌러야 합니다.
아래는 **웹캠 영상을 실시간으로 계속 받아오면서** 매 프레임마다 탐지 결과를
오버레이로 겹쳐 보여주는, 조금 더 복잡한 버전입니다.

동작 방식: 브라우저에 보이지 않는 `<canvas>`로 매 프레임을 캡처해 Python으로
보내면, Python이 YOLO로 탐지한 뒤 bounding box만 그려진 **투명 PNG**를 다시
브라우저로 보내 비디오 위에 겹쳐 그립니다. 비디오 프레임 **자체**는 브라우저
밖으로 나가지 않고(캡처 → 탐지 → 오버레이) 계속 반복되므로, 눈으로 보기에는
실시간으로 탐지되는 것처럼 보입니다.

**주의할 점**
- 이 방식은 브라우저/네트워크 상황에 따라 위의 사진 캡처 버전보다 불안정할 수
  있습니다. 안 되면 위의 버전을 사용하세요.
- 비디오 영상이나 "종료하려면..." 문구를 클릭하면 스트리밍이 멈춥니다.

In [ ]:
try:
    from google.colab.output import eval_js
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("이 데모도 Google Colab에서 브라우저 웹캠에 접근해야 동작합니다. Colab에서 실행해주세요.")
else:
    import base64
    import io

    from IPython.display import display, Javascript
    from PIL import ImageDraw

    VIDEO_STREAM_JS = '''
        var video;
        var div = null;
        var stream;
        var captureCanvas;
        var imgElement;
        var labelElement;

        var pendingResolve = null;
        var shutdown = false;

        function removeDom() {
          stream.getVideoTracks()[0].stop();
          video.remove();
          div.remove();
          video = null;
          div = null;
          stream = null;
          imgElement = null;
          captureCanvas = null;
          labelElement = null;
        }

        function onAnimationFrame() {
          if (!shutdown) {
            window.requestAnimationFrame(onAnimationFrame);
          }
          if (pendingResolve) {
            var result = "";
            if (!shutdown) {
              captureCanvas.getContext("2d").drawImage(video, 0, 0, 640, 480);
              result = captureCanvas.toDataURL("image/jpeg", 0.8);
            }
            var lp = pendingResolve;
            pendingResolve = null;
            lp(result);
          }
        }

        async function createDom() {
          if (div !== null) {
            return stream;
          }

          div = document.createElement("div");
          div.style.border = "2px solid black";
          div.style.padding = "3px";
          div.style.width = "100%";
          div.style.maxWidth = "640px";
          document.body.appendChild(div);

          const modelOut = document.createElement("div");
          modelOut.innerHTML = "<span>Status:</span>";
          labelElement = document.createElement("span");
          labelElement.innerText = "No data";
          labelElement.style.fontWeight = "bold";
          modelOut.appendChild(labelElement);
          div.appendChild(modelOut);

          video = document.createElement("video");
          video.style.display = "block";
          video.width = div.clientWidth - 6;
          video.setAttribute("playsinline", "");
          video.onclick = () => { shutdown = true; };
          stream = await navigator.mediaDevices.getUserMedia({ video: true });
          div.appendChild(video);

          imgElement = document.createElement("img");
          imgElement.style.position = "absolute";
          imgElement.style.zIndex = 1;
          imgElement.onclick = () => { shutdown = true; };
          div.appendChild(imgElement);

          const instruction = document.createElement("div");
          instruction.innerHTML =
              "<span style='color: red; font-weight: bold;'>" +
              "종료하려면 비디오나 이 문구를 클릭하세요.</span>";
          div.appendChild(instruction);
          instruction.onclick = () => { shutdown = true; };

          video.srcObject = stream;
          await video.play();

          captureCanvas = document.createElement("canvas");
          captureCanvas.width = 640;
          captureCanvas.height = 480;
          window.requestAnimationFrame(onAnimationFrame);

          return stream;
        }

        async function stream_frame(label, imgData) {
          if (shutdown) {
            removeDom();
            shutdown = false;
            return "";
          }

          stream = await createDom();

          if (label != "") {
            labelElement.innerHTML = label;
          }

          if (imgData != "") {
            var videoRect = video.getClientRects()[0];
            imgElement.style.top = videoRect.top + "px";
            imgElement.style.left = videoRect.left + "px";
            imgElement.style.width = videoRect.width + "px";
            imgElement.style.height = videoRect.height + "px";
            imgElement.src = imgData;
          }

          var result = await new Promise(function(resolve, reject) {
            pendingResolve = resolve;
          });
          shutdown = false;

          return result;
        }
    '''

    def start_stream():
        display(Javascript(VIDEO_STREAM_JS))

    def call_stream_frame(label, overlay_data_url):
        return eval_js(f'stream_frame("{label}", "{overlay_data_url}")')

    def js_to_image(js_reply):
        image_bytes = base64.b64decode(js_reply.split(",")[1])
        return Image.open(io.BytesIO(image_bytes)).convert("RGB")

    def overlay_to_data_url(overlay):
        buf = io.BytesIO()
        overlay.save(buf, format="PNG")
        encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
        return f"data:image/png;base64,{encoded}"

    def draw_detections_overlay(result, width=640, height=480):
        """탐지된 box만 그려진 투명 PNG를 만듭니다 (배경은 완전히 투명합니다)."""
        overlay = Image.new("RGBA", (width, height), (0, 0, 0, 0))
        draw = ImageDraw.Draw(overlay)
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            conf = box.conf.item()
            cls_id = int(box.cls.item())
            label_text = f"{result.names[cls_id]} {conf:.0%}"
            draw.rectangle([x1, y1, x2, y2], outline=(0, 255, 0, 255), width=3)
            draw.text((x1, max(0, y1 - 14)), label_text, fill=(0, 255, 0, 255))
        return overlay

    start_stream()
    overlay_data_url = ""
    label_html = ""
    while True:
        js_reply = call_stream_frame(label_html, overlay_data_url)
        if not js_reply:
            break  # 비디오나 안내 문구를 클릭하면 여기서 멈춥니다.

        frame = js_to_image(js_reply)
        stream_result = yolo_model.predict(source=frame, conf=0.25, verbose=False)[0]

        overlay = draw_detections_overlay(stream_result)
        overlay_data_url = overlay_to_data_url(overlay)
        label_html = f"{len(stream_result.boxes)} detections"

In [ ]:
# 메모리 정리
del yolo_model
if device.type == "cuda":
    torch.cuda.empty_cache()

## 정리

이번 노트북에서는 CNN을 이용한 컴퓨터 비전의 세 가지 핵심 작업을 실습했습니다.

- **Part 1 (이미지 분류)**: ImageNet으로 사전학습된 ResNet18을 그대로 가져와
  이미지가 1,000개 클래스 중 무엇에 가까운지 예측했습니다. 모델을 전혀 학습시키지
  않고도 바로 쓸 수 있다는 것이 사전학습 모델의 힘입니다.
- **Part 2 (전이학습)**: ImageNet 1,000개 클래스에는 없는 "고양이/개 37개 품종"
  문제에 같은 사전학습 가중치를 재활용했습니다. 마지막 층만 학습시키는
  **Feature Extraction**과, 뒤쪽 일부 층까지 함께 학습시키는 **Fine-tuning**을
  비교해, 적은 데이터로도 새 문제에 적응시킬 수 있음을 확인했습니다.
- **Part 3 (객체 탐지)**: "무엇이 있는가"뿐 아니라 "어디에 있는가"까지 답하는
  YOLO로 이미지에서 여러 물체를 동시에 탐지했습니다. confidence threshold와
  NMS(IoU threshold)가 결과에 어떤 영향을 주는지 확인한 뒤, 마지막으로 여러분의
  웹캠으로 직접 찍은 사진에도 같은 모델을 돌려봤습니다.

세 Part 모두 사전학습된 모델이 핵심이었습니다 — 거대한 데이터셋으로 미리 학습된
지식을 그대로 쓰거나(Part 1, 3), 조금만 다시 학습시켜 재활용(Part 2)하는 것이
현실적인 컴퓨터 비전 문제를 푸는 가장 실용적인 방법입니다.